# RSNA 2018 — Dados, pré-processamento e features (HOG / LBP / GLCM)
Parte do Ruan. Antes de rodar: entre em https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge e aceite as regras (*Join Competition*).
Gera `features_all.csv` (imagem inteira) e `features_regions.csv` (regiões candidatas).

### 1. Repositório e dependências
Clona a branch com o código (mesmo código do repositório, sem cópia solta). Ajuste `BRANCH` se o nome for outro.

In [ ]:
BRANCH = "ruan"
!git clone -q -b {BRANCH} https://github.com/erik-hp/ap1-rsna-pneumonia-baseline.git repo
%cd repo
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install kaggle

### 2. Credenciais do Kaggle
Kaggle → Settings → API → *Generate New Token* (ou use o `kaggle.json` antigo na alternativa comentada).

In [ ]:
import os, getpass
os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Kaggle API token: ")

# Alternativa (kaggle.json):
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

### 3. Download e extração (só treino estágio 2)
Se o nome do .zip ou da competição for diferente, copie o comando da aba *Data* do desafio.

In [ ]:
!kaggle competitions download -c rsna-pneumonia-detection-challenge -p data
!mkdir -p data/raw && unzip -q data/rsna-pneumonia-detection-challenge.zip "stage_2_train_images/*" stage_2_train_labels.csv -d data/raw
!ls data/raw/stage_2_train_images | wc -l

### 4. Testes rápidos do pipeline

In [ ]:
!pytest -q tests/test_features.py

### 5. Extração (seed 42, amostra estratificada por classe)
Teste antes com `--n-per-class 50` para medir o tempo.

In [ ]:
!python scripts/extract_features.py --raw data/raw --out data/processed --sampling natural --n-total 2000

### 6. Validação antes de entregar

In [ ]:
import pandas as pd, numpy as np
for f in ["features_all", "features_regions"]:
    d = pd.read_csv(f"data/processed/{f}.csv")
    X = d.drop(columns=[c for c in ["patientId", "Target", "x", "y", "width", "height"] if c in d])
    assert np.isfinite(X.values).all() and d.patientId.notna().all() and set(d.Target.unique()) <= {0, 1}
    assert {c.split("_")[0] for c in X.columns} <= {"hog", "lbp", "glcm"}
    print(f, d.shape, d.Target.value_counts().to_dict())

### 7. Salvar no Google Drive (o disco do Colab é apagado ao desconectar)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/rsna && cp data/processed/*.csv /content/drive/MyDrive/rsna/